In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_data.csv",
                 parse_dates=["StartDatetime", "EndDatetime", "StartDay", "EndDay"])
df = df.sort_values("StartDatetime").reset_index(drop=True)   # must be time-ordered
print(df.shape)

(72843, 13)


In [4]:
df["Hour"] = df["StartDatetime"].dt.hour
df["Day"] = df["StartDatetime"].dt.day
df["Month"] = df["StartDatetime"].dt.month
df["DayOfWeek"] = df["StartDatetime"].dt.dayofweek      # 0 = Monday
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)

def time_of_day(h):
    if 5 <= h < 12: return "Morning"
    if 12 <= h < 17: return "Afternoon"
    if 17 <= h < 21: return "Evening"
    return "Night"

df["TimeOfDay"] = df["Hour"].apply(time_of_day)

In [5]:
# how many sessions the user had before this one
df["User_PrevSessions"] = df.groupby("UserID").cumcount()
df["User_AvgPastDemand"] = (df.groupby("UserID")["Demand"]
                              .transform(lambda s: s.shift(1).expanding().mean()))

# same for the charger
df["Charger_PrevSessions"] = df.groupby("ChargerID").cumcount()
df["Charger_AvgPastDemand"] = (df.groupby("ChargerID")["Demand"]
                                 .transform(lambda s: s.shift(1).expanding().mean()))

# first-time users/chargers have no history
df["IsNewUser"] = (df["User_PrevSessions"] == 0).astype(int)
df[["User_AvgPastDemand", "Charger_AvgPastDemand"]] = \
    df[["User_AvgPastDemand", "Charger_AvgPastDemand"]].fillna(0)

In [6]:
median_demand = df["Demand"].median()
df["HighDemand"] = (df["Demand"] >= median_demand).astype(int)
print("Median:", median_demand)
print(df["HighDemand"].value_counts(normalize=True))   # check imbalance here

Median: 14.1
HighDemand
1    0.500432
0    0.499568
Name: proportion, dtype: float64


In [7]:
print(df.isnull().sum().sum(), "nulls")
print(df.head())
df.to_csv("../data/processed/featured_data.csv", index=False)

0 nulls
   UserID  ChargerID  ChargerCompany            Location  ChargerType  \
0     102        291               1  public institution            0   
1     104        191               1              resort            0   
2     105        320               1             camping            0   
3       0        433               1             camping            0   
4     391        250               1         public area            0   

    StartDay StartTime     EndDay  EndTime       StartDatetime  ... Month  \
0 2021-09-30  17:14:26 2021-10-01  1:21:28 2021-09-30 17:14:00  ...     9   
1 2021-09-30  18:22:36 2021-10-01  2:21:03 2021-09-30 18:22:00  ...     9   
2 2021-09-30  18:57:46 2021-10-01  0:44:59 2021-09-30 18:57:00  ...     9   
3 2021-09-30  19:52:22 2021-10-01  0:21:51 2021-09-30 19:52:00  ...     9   
4 2021-09-30  20:48:46 2021-10-01  3:35:11 2021-09-30 20:48:00  ...     9   

   DayOfWeek  IsWeekend  TimeOfDay  User_PrevSessions  User_AvgPastDemand  \
0          3 